In [2]:
# -*- coding: utf-8 -*-
import os
from pathlib import Path
from typing import List, Optional
import pandas as pd
import pyarrow as pya
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import glob
from itertools import chain
import numpy as np
from collections import deque, defaultdict
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os, urllib.parse, urllib.request

In [33]:
import os, time, traceback, urllib.parse, urllib.request
from contextlib import contextmanager
from urllib.error import HTTPError, URLError

def notify_cell(message: str) -> bool:
    """Send a simple Telegram message."""
    token = os.getenv("TELEGRAM_BOT_TOKEN")
    chat_id = os.getenv("TELEGRAM_CHAT_ID")
    if not token or not chat_id:
        print("❌ Missing TELEGRAM_BOT_TOKEN or TELEGRAM_CHAT_ID")
        return False

    url = f"https://api.telegram.org/bot{token}/sendMessage"
    data = urllib.parse.urlencode({"chat_id": chat_id, "text": message}).encode()

    try:
        with urllib.request.urlopen(urllib.request.Request(url, data=data), timeout=20) as r:
            print(f"📨 Sent: {message}")
            return True
    except (HTTPError, URLError) as e:
        print("🚫 Telegram error:", e)
        return False


@contextmanager
def notify_wrap(task_name: str):
    """Context manager to auto-notify on success or failure."""
    start = time.time()
    try:
        yield
    except Exception as e:
        elapsed = time.time() - start
        err_text = ''.join(traceback.format_exception_only(type(e), e)).strip()
        notify_cell(
            f"❌ FAILED: {task_name}\n"
            f"Error: `{err_text}`\n"
            f"Duration: {elapsed/60:.1f} min"
        )
        raise
    else:
        elapsed = time.time() - start
        notify_cell(
            f"✅ DONE: {task_name}\n"
            f"Duration: {elapsed/60:.1f} min"
        )



In [ ]:
df1 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump1.parquet')
df2 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump2.parquet')
df3 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump3.parquet')
df4 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump4.parquet')
df5 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump5.parquet')
df7 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump7.parquet')

dump1 = df1.to_pandas()
dump2 = df2.to_pandas()
dump3 = df3.to_pandas()
dump4 = df4.to_pandas()
dump5 = df5.to_pandas()
dump6 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump6.parquet').to_pandas()
dump7 = df7.to_pandas()

# Load attack dumps
#attack_044h = pq.read_table(r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-masq-044h.parquet").to_pandas()
#attack_080h = pq.read_table(r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-masq-080h.parquet").to_pandas()
#attack_383h = pq.read_table(r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-masq-383h.parquet").to_pandas()
#attack_593h = pq.read_table(r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-masq-593h.parquet").to_pandas()"""


In [35]:
files_pathname = r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw"
masq_files = [
    f for f in os.listdir(files_pathname)
    if f.startswith("dump6-masq-") and f.endswith(".parquet")
]

masq_ids = [f.split('-')[-1].removesuffix('.parquet') for f in masq_files]

print(f"Found {len(masq_ids)} masquerade files.")
print("Masquerade IDs in discovery order:", masq_ids)

Found 35 masquerade files.
Masquerade IDs in discovery order: ['044h', '080h', '081h', '111h', '112h', '113h', '162h', '18Fh', '200h', '220h', '251h', '260h', '2B0h', '316h', '329h', '381h', '383h', '386h', '387h', '47Fh', '4F1h', '50Ch', '52Ah', '541h', '545h', '547h', '549h', '553h', '555h', '556h', '557h', '58Bh', '593h', '5A0h', '5B0h']


In [36]:
with notify_wrap("Cell 1: hex to bytes and hamming distance computation"):
    def hex_to_bytes(h):
        
        if h is None or (isinstance(h, float) and np.isnan(h)): 
            return b""
        if isinstance(h, (bytes, bytearray)): 
            return bytes(h)
        s = str(h).strip().replace("0x","").replace(" ","")
        if s == "": 
            return b""
        if len(s) % 2 == 1: 
            s = "0"+s
        try: 
            return bytes.fromhex(s)
        except Exception: 
            return str(h).encode("utf-8", errors="ignore")


    def hamming_distance(a: bytes, b: bytes) -> int:
       
        len_mismatch = (len(a) != len(b))
    
        # Pad to same length
        max_len = max(len(a), len(b))
        a_padded = a + b'\x00' * (max_len - len(a))
        b_padded = b + b'\x00' * (max_len - len(b))
        
        # Count differences
        #dist = sum(byte_a != byte_b for byte_a, byte_b in zip(a_padded, b_padded))
        
        distance = 0
        for byte_a, byte_b in zip(a_padded, b_padded):
            distance += bin(byte_a ^ byte_b).count('1') # Count 1s in binary representation
        return (distance)

📨 Sent: ✅ DONE: Cell 1: hex to bytes and hamming distance computation
Duration: 0.0 min


In [37]:
with notify_wrap("Cell 1: compute Hamming distances of dumps"):
    def compute_hamming_distances(dumps, out_csv, ranges, process_per_dump=True):
        
        Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
        
        #records keep track of all values of hamming distance for each id so i can group by them later
        records = []
        prev_payload = {}  # {arbitration_id: previous_bytes} i can use this to store the previous payload for each id
        ranges_indexed = ranges.set_index('arbitration_id') if ranges is not None else None
        
        #for each dump that i pass
        for dump_name, df in dumps:
            #copy it
            d = df.copy()
            
            # Ensure timestamp column and eventually rename it
            if "timestamp" not in d.columns:
                if d.index.name == "timestamp":
                    d = d.reset_index()
                else:
                    d = d.reset_index().rename(columns={"index": "timestamp"})
            
            #sort by timestamp and check if label exists
            d = d.sort_values("timestamp", kind="mergesort")
            has_label = "label" in d.columns
            
            #for each row save id, timestamp, the current data and label if exists
            for _, row in d.iterrows():
                aid = row["arbitration_id"]
                ts = row["timestamp"]
                lab = int(row["label"]) if has_label and not pd.isna(row["label"]) else 0
                curr_bytes = hex_to_bytes(row["data"])
                
                #get the previous payload for this specific ID
                prev = prev_payload.get(aid)
                
                #if i have a previous id so its not the first time this id appears then compute the distance and append it to the records
                if prev is not None:
                    should_update = False
                    # Compute Hamming distance
                    dist = hamming_distance(curr_bytes, prev)
                    #max_len = max(len(curr_bytes), len(prev))
                    #norm_dist = dist / max_len if max_len > 0 else 0.0
                    #min_val = None
                    #max_val = None
                    
                    try:
                        if ranges_indexed is not None and aid in ranges_indexed.index:
                            min_val = ranges_indexed.at[aid, 'min_hamming']
                            max_val = ranges_indexed.at[aid, 'max_hamming']
                    except KeyError:
                        pass
                    
                    if min_val is not None and max_val is not None:
                        if min_val <= dist <= max_val:
                            should_update = True
                        #if outside range = suspicious, don't update
                        else:
                            #unknown ID (not in training) - don't update
                            should_update = False
                
                    
                    
                    records.append({
                        "dump": dump_name,
                        "timestamp": ts,
                        "arbitration_id": aid,
                        "payload_len": len(curr_bytes),
                        "ham_dist": dist,
                        #"ham_norm": norm_dist,
                        "label": lab
                    })
                    if should_update:   
                    # Update previous payload
                        prev_payload[aid] = curr_bytes
               
                prev_payload[aid] = curr_bytes
            #if this is true i can clear the previous payloads so that each dump is processed independently but i will put it to false for the benign dumps so i can use them all together
            if process_per_dump:
                prev_payload.clear()
        
        out_df = pd.DataFrame.from_records(records)
        out_df.to_csv(out_csv, index=False)
        print(f"Saved: {out_csv} (rows={len(out_df):,})")
        
        return out_df

📨 Sent: ✅ DONE: Cell 1: compute Hamming distances of dumps
Duration: 0.0 min


In [38]:
with notify_wrap("Cell 2: build_hamming_range_model"):
    def build_hamming_range_model(hamming_csv, output_csv):
        
        
        df = pd.read_csv(hamming_csv)
        
        # Group by ID and compute min/max
        ranges = (df.groupby('arbitration_id')['ham_dist']
                .agg(['min', 'max', 'count'])
                .reset_index())
        
        ranges.columns = ['arbitration_id', 'min_hamming', 'max_hamming', 'n_samples']
        
        # Compute range size
        ranges['range_size'] = ranges['max_hamming'] - ranges['min_hamming']
        
        # Normalized versions (for 8-byte payloads, max=8)
        #ranges['min_norm'] = ranges['min_hamming'] / 8.0
        #ranges['max_norm'] = ranges['max_hamming'] / 8.0
        #ranges['range_norm'] = ranges['range_size'] / 8.0
        
        # Classify IDs based on sigma, i chose 6 as per the paper but then i switched over to 24 for testing. 
        # not  useful for detection but just to have some metrics
        sigma = 24
        """def classify(r):
            if r == 0:
                return 'NoRange'
            elif r <= sigma:
                return 'SmallRange'
            else:
                return 'MidRange'"""
        
        #ranges['class'] = ranges['range_size'].apply(classify)
        
        # Expected detection rates 
        """def expected_tpr(cls):
            if cls == 'NoRange':
                return 0.98  
            elif cls == 'SmallRange':
                return 0.97  
            else:
                return 0.25  
        
        ranges['expected_tpr'] = ranges['class'].apply(expected_tpr)
        
        # Sort by range size
        #ranges = ranges.sort_values('range_size')"""
        
        # Save
        Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
        ranges.to_csv(output_csv, index=False)
        
        """# Print summary
        
        print(f"\n Range Classification (σ={sigma}):")
        class_counts = ranges['class'].value_counts()
        for cls in ['NoRange', 'SmallRange', 'MidRange']:
            if cls in class_counts.index:
                count = class_counts[cls]
                pct = count / len(ranges) * 100
                print(f"   {cls:12s}: {count:3d} IDs ({pct:5.1f}%)")
        
        print(f"\n Range Statistics:")
        print(f"   Mean range:   {ranges['range_size'].mean():.2f} bytes")
        print(f"   Median range: {ranges['range_size'].median():.2f} bytes")
        print(f"   Max range:    {ranges['range_size'].max():.0f} bytes")
        """
        return ranges


📨 Sent: ✅ DONE: Cell 2: build_hamming_range_model
Duration: 0.0 min


In [39]:
with notify_wrap("Cell 3: simple detection with Hamming distances"):
    def detect_simple(test_csv, ranges_csv, output_csv="artifacts/detections_simple.csv"):
        
        
        # Load data
        test = pd.read_csv(test_csv)
        ranges = pd.read_csv(ranges_csv)
        
        # Merge test data with ranges
        test_with_ranges = test.merge(
            ranges[['arbitration_id', 'min_hamming', 'max_hamming', 'range_size']],
            on='arbitration_id',
            how='left'
        )
        
        # Detect: Flag if OUTSIDE range [min, max]
        test_with_ranges['is_anomaly'] = (
            (test_with_ranges['ham_dist'] < test_with_ranges['min_hamming']) |
            (test_with_ranges['ham_dist'] > test_with_ranges['max_hamming'])
        )
        
        # Count
        n_anomalies = test_with_ranges['is_anomaly'].sum()
        n_total = len(test_with_ranges)
        
        print(f"\n Detection Summary:")
        print(f"   Total messages:    {n_total:,}")
        print(f"   Flagged anomalies: {n_anomalies:,} ({n_anomalies/n_total*100:.2f}%)")
        
        # Compute metrics if labels available
        metrics = {}
        if 'label' in test_with_ranges.columns:
            y_true = test_with_ranges['label'].values
            y_pred = test_with_ranges['is_anomaly'].values
            
            TP = ((y_true == 1) & (y_pred == True)).sum()
            FP = ((y_true == 0) & (y_pred == True)).sum()
            TN = ((y_true == 0) & (y_pred == False)).sum()
            FN = ((y_true == 1) & (y_pred == False)).sum()
            
            tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
            fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
            precision = TP / (TP + FP) if (TP + FP) > 0 else 0
            f1 = 2 * precision * tpr / (precision + tpr) if (precision + tpr) > 0 else 0
            
            print(f"\n Confusion Matrix:")
            print(f"   TP (attacks detected):  {TP:,}")
            print(f"   FP (false alarms):      {FP:,}")
            print(f"   TN (benign correct):    {TN:,}")
            print(f"   FN (attacks missed):    {FN:,}")
            
            print(f"\n Performance Metrics:")
            print(f"   TPR (Detection Rate): {tpr*100:.2f}%")
            print(f"   FPR (False Alarms):   {fpr*100:.2f}%")
            print(f"   Precision:            {precision*100:.2f}%")
            print(f"   F1-Score:             {f1*100:.2f}%")
            
            metrics['overall'] = {
                'TP': int(TP), 'FP': int(FP), 'TN': int(TN), 'FN': int(FN),
                'TPR': tpr, 'FPR': fpr, 'Precision': precision, 'F1': f1
            }
            
            """for cls in ['NoRange', 'SmallRange', 'MidRange']:
                cls_data = test_with_ranges[test_with_ranges['class'] == cls]
                if len(cls_data) > 0 and 'label' in cls_data.columns:
                    cls_attacks = cls_data[cls_data['label'] == 1]
                    cls_detected = cls_attacks[cls_attacks['is_anomaly']]
                    
                    if len(cls_attacks) > 0:
                        actual_tpr = len(cls_detected) / len(cls_attacks)
                        expected_tpr = cls_data['expected_tpr'].iloc[0]
                        n_ids = cls_data['arbitration_id'].nunique()
                        
                        print(f"   {cls:12s} {expected_tpr*100:>6.1f}%      {actual_tpr*100:>6.1f}%      {n_ids:>3d}")
            
            # Per-ID breakdown
            per_id = []
            for aid in test_with_ranges['arbitration_id'].unique():
                aid_data = test_with_ranges[test_with_ranges['arbitration_id'] == aid]
                
                if 'label' not in aid_data.columns:
                    continue
                
                y_t = aid_data['label'].values
                y_p = aid_data['is_anomaly'].values
                
                tp = ((y_t == 1) & (y_p == True)).sum()
                fn = ((y_t == 1) & (y_p == False)).sum()
                tpr_id = tp / (tp + fn) if (tp + fn) > 0 else 0
                
                aid_class = aid_data['class'].iloc[0] if len(aid_data) > 0 else 'Unknown'
                aid_range = aid_data['range_size'].iloc[0] if len(aid_data) > 0 else np.nan
                
                per_id.append({
                    'arbitration_id': aid,
                    'aid_hex': f"0x{aid:03X}",
                    'n_attacks': int(tp + fn),
                    'detected': int(tp),
                    'missed': int(fn),
                    'tpr': tpr_id,
                    'class': aid_class,
                    'range_size': aid_range
                })
            
            metrics['per_id'] = pd.DataFrame(per_id)"""
        
        # Save results
        Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
        test_with_ranges.to_csv(output_csv, index=False)
        print(f"\n Results saved to: {output_csv}")
        
        return test_with_ranges, metrics


📨 Sent: ✅ DONE: Cell 3: simple detection with Hamming distances
Duration: 0.0 min


In [40]:

if __name__ == "__main__":
    print("\n" + "="*70)
    print("HAMMING DISTANCE IDS")
    print("="*70)
    
    """benign_dumps = [
        ("dump1", dump1),
        ("dump2", dump2),
        ("dump3", dump3),
        ("dump4", dump4),
        ("dump5", dump5),
        ("dump6", dump6),
        ("dump7", dump7)
    ]"""
    """benign_ham = compute_hamming_distances(
    benign_dumps, 
    "artifacts/benign_hamming.csv",
    process_per_dump=False  
)"""

    
    results_summary = []
    
    ranges = build_hamming_range_model(
    "artifacts/benign_hamming.csv",
    "artifacts/hamming_ranges_masq.csv"
)
    print("\nHamming Ranges Model Built.")
    
    for i, masq_id in enumerate(masq_ids, 1):
        print(f"\n[{i}/{len(masq_ids)}] Testing: dump6-masq-{masq_id}")
        
        try:
            # Load attack file
            attack_file = os.path.join(files_pathname, f"dump6-masq-{masq_id}.parquet")
            attack_df = pq.read_table(attack_file).to_pandas()
            
            # Compute Hamming
            attack_dumps = [(f"masq_{masq_id}", attack_df)]
            attack_ham = compute_hamming_distances(
                attack_dumps,
                f"artifacts/attack_ham_{masq_id}.csv",
                ranges,
                process_per_dump=True
            )
            
            # Detect
            results, metrics = detect_simple(
                f"artifacts/attack_ham_{masq_id}.csv",
                "artifacts/hamming_ranges_masq.csv",
                f"artifacts/detections_{masq_id}.csv"
            )
            
            # Store results
            if 'overall' in metrics:
                target_aid = int(masq_id.replace('h', ''), 16)
                results_summary.append({
                    'masq_id': masq_id,
                    'target_aid_hex': f"0x{target_aid:03X}",
                    'TPR': metrics['overall']['TPR'] * 100,
                    'FPR': metrics['overall']['FPR'] * 100,
                    'F1': metrics['overall']['F1'] * 100,
                    'n_attacks': metrics['overall']['TP'] + metrics['overall']['FN']
                })
                
                print(f"   ✅ TPR: {metrics['overall']['TPR']*100:.1f}%, "
                      f"FPR: {metrics['overall']['FPR']*100:.2f}%")
                notify_cell(f"Finished analyzing masquerade ID: {masq_id}" + f" number [{i}/{len(masq_ids)}]" + f"\nTPR: {metrics['overall']['TPR']*100:.1f}%, FPR: {metrics['overall']['FPR']*100:.2f}%"+ f"\nF1-Score: {metrics['overall']['F1']*100:.2f}%")
        
        except Exception as e:
            print(f"Error: {e}")
            continue
    
    # Create summary
    summary_df = pd.DataFrame(results_summary)
    summary_df.to_csv("artifacts/all_masq_summary.csv", index=False)
    
    print("\n" + "="*70)
    print("FINAL SUMMARY")
    print("="*70)
    print(f"\nTested: {len(summary_df)}/{len(masq_ids)} attack types")
    print(f"\nOverall Performance:")
    print(f"   Mean TPR:    {summary_df['TPR'].mean():.2f}%")
    print(f"   Mean FPR:    {summary_df['FPR'].mean():.4f}%")
    print(f"   Mean F1:     {summary_df['F1'].mean():.2f}%")
    
    print("\nResults by Attack Type:")
    print(summary_df.to_string(index=False))
    print("="*70)


HAMMING DISTANCE IDS

Hamming Ranges Model Built.

[1/35] Testing: dump6-masq-044h
Saved: artifacts/attack_ham_044h.csv (rows=4,279,845)

 Detection Summary:
   Total messages:    4,279,845
   Flagged anomalies: 1,029 (0.02%)

 Confusion Matrix:
   TP (attacks detected):  1,028
   FP (false alarms):      1
   TN (benign correct):    4,278,816
   FN (attacks missed):    0

 Performance Metrics:
   TPR (Detection Rate): 100.00%
   FPR (False Alarms):   0.00%
   Precision:            99.90%
   F1-Score:             99.95%

 Results saved to: artifacts/detections_044h.csv
   ✅ TPR: 100.0%, FPR: 0.00%
📨 Sent: Finished analyzing masquerade ID: 044h number [1/35]
TPR: 100.0%, FPR: 0.00%
F1-Score: 99.95%

[2/35] Testing: dump6-masq-080h
Saved: artifacts/attack_ham_080h.csv (rows=4,279,845)

 Detection Summary:
   Total messages:    4,279,845
   Flagged anomalies: 96,010 (2.24%)

 Confusion Matrix:
   TP (attacks detected):  96,009
   FP (false alarms):      1
   TN (benign correct):    4,183,

In [11]:
masq_df = pd.read_csv("artifacts/attack_ham_112h.csv")  # The one with always update
print("First 50 packets for ID 0x112:")
print(masq_df[masq_df['arbitration_id'] == 274][['ham_dist', 'label']].head(50))

First 50 packets for ID 0x112:
      ham_dist  label
2            3      0
20           3      0
39           3      0
59           3      0
79           3      0
95           3      0
116          3      0
136          3      0
155          3      0
176          3      0
194          3      0
214          3      0
235          3      0
256          3      0
275          3      0
296          3      0
320          3      0
343          3      0
365          3      0
387          3      0
409          3      0
429          3      0
448          3      0
468          3      0
490          3      0
507          3      0
532          3      0
556          3      0
582          3      0
603          3      0
621          3      0
643          3      0
664          3      0
683          3      0
703          3      0
722          3      0
748          3      0
771          3      0
795          3      0
822          3      0
842          3      0
862          3      0
881          3      0
9

In [15]:
attack_file = pq.read_table("X-CANIDS/raw/dump6-masq-112h.parquet").to_pandas()
print("Unique IDs in masq-112h file:")
print(attack_file['arbitration_id'].value_counts())

print("\nAttacks per ID:")
print(attack_file[attack_file['label'] == 1]['arbitration_id'].value_counts())

Unique IDs in masq-112h file:
arbitration_id
128     196211
129     196211
512     196211
809     196211
790     196211
         ...  
1456      1963
1530      1960
1407       981
2024         4
2016         2
Name: count, Length: 64, dtype: int64

Attacks per ID:
arbitration_id
274    95998
Name: count, dtype: int64


In [7]:
# Load the detection results WITH should_update
masq_data = pd.read_csv("artifacts/detections_112h.csv")

print("Attack pattern analysis:")
attacks = masq_data[masq_data['label'] == 1]
print(f"Total attacks: {len(attacks)}")
print(f"Detected attacks: {attacks['is_anomaly'].sum()}")
print(f"\nAttack distances:")
print(attacks['ham_dist'].value_counts().head(20))

print("\n\nFalse Positive analysis:")
fps = masq_data[(masq_data['label'] == 0) & (masq_data['is_anomaly'] == True)]
print(f"Total FPs: {len(fps)}")
print(f"\nFP distances:")
print(fps['ham_dist'].value_counts().head(20))

print("\n\nRange for this ID:")
print(f"Min: {attacks['min_hamming'].iloc[0]}")
print(f"Max: {attacks['max_hamming'].iloc[0]}")

Attack pattern analysis:
Total attacks: 95998
Detected attacks: 95998

Attack distances:
ham_dist
64    95997
49        1
Name: count, dtype: int64


False Positive analysis:
Total FPs: 1

FP distances:
ham_dist
21    1
Name: count, dtype: int64


Range for this ID:
Min: 0
Max: 15


In [ ]:
notify_cell("🚀 Hamming Distance IDS Pipeline Completed Successfully!")

📨 Sent: 🚀 Hamming Distance IDS Pipeline Completed Successfully!


True

In [9]:
ranges_df = pd.read_csv("artifacts/hamming_ranges_masq.csv")
id_112 = int('112', 16)
print(f"ID 0x112 = {id_112}")
print(f"Is ID in ranges? {id_112 in ranges_df['arbitration_id'].values}")
print(ranges_df[ranges_df['arbitration_id'] == id_112])

ID 0x112 = 274
Is ID in ranges? True
   arbitration_id  min_hamming  max_hamming  n_samples  range_size
7             274            0           15    1250040          15
